In [1]:
pip install "elasticsearch>=8.15,<9"

  Using cached elastic_transport-8.19.0-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/952.8

## Connect to Elasticsearch

The cluster runs locally in Docker on `http://localhost:9200` with security disabled (no username/password, no TLS -- confirmed via `curl http://localhost:9200/`). `es.info()` below double-checks connectivity before doing anything else.

In [5]:
import json
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ES_HOST = "http://localhost:9200"
INDEX_NAME = "document-chunks"
EMBEDDED_JSON_PATH = Path("embedded_chunks.json")

es = Elasticsearch(ES_HOST)

info = es.info()
print(f"Connected to {info['cluster_name']} (Elasticsearch {info['version']['number']})")

Connected to docker-cluster (Elasticsearch 8.15.0)


## Index mapping

Flat documents (no `nested`/`join`) -- the book/chapter/section hierarchy is just low-cardinality metadata, so plain fields + filters on `chapter_number`/`section_index` are simpler and faster than a join structure.

- `chunk_id`, `book_id`, `author`, `chapter_title`, `prev_chunk_id`, `next_chunk_id` -> `keyword` (exact-match/filter fields, never analyzed).
- `book_title` -> `text` + a `.keyword` sub-field, so it supports both full-text and exact-match/aggregation.
- `chapter_number`, `section_index`, `chunk_index`, `global_index`, `paragraph_ids`, `new_paragraph_ids`, `token_count` -> `integer` (arrays of integers work natively in ES with no special array type).
- `text` -> `text` with the `english` analyzer, for BM25/highlighting.
- `embedding` -> `dense_vector`, `dims: 1024`, `similarity: cosine` (bge-m3's dense output is already L2-normalized), indexed with HNSW for fast kNN.
- `sparse_embedding` -> `rank_features`, not `sparse_vector`. `sparse_vector` in this ES version is built around ELSER/inference pipelines; our sparse weights are computed ourselves by bge-m3 (not via an ES inference endpoint), and `rank_features` accepts arbitrary token -> positive-float pairs directly with no inference dependency -- a simpler, stabler fit here. It requires positive weights, which bge-m3's lexical weights already are.
- `number_of_shards: 1`, `number_of_replicas: 0` -- a single-node local cluster with ~250 tiny documents doesn't need sharding or replicas; this also avoids a permanently yellow cluster health from unassigned replica shards.

In [6]:
INDEX_MAPPING = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "book_id": {"type": "keyword"},
            "book_title": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
            "author": {"type": "keyword"},
            "chapter_number": {"type": "integer"},
            "chapter_title": {"type": "keyword"},
            "section_index": {"type": "integer"},
            "chunk_index": {"type": "integer"},
            "global_index": {"type": "integer"},
            "type": {"type": "keyword"},
            "paragraph_ids": {"type": "integer"},
            "new_paragraph_ids": {"type": "integer"},
            "token_count": {"type": "integer"},
            "prev_chunk_id": {"type": "keyword"},
            "next_chunk_id": {"type": "keyword"},
            "text": {"type": "text", "analyzer": "english"},
            "embedding": {
                "type": "dense_vector",
                "dims": 1024,
                "index": True,
                "similarity": "cosine",
                "index_options": {"type": "hnsw", "m": 16, "ef_construction": 100},
            },
            "sparse_embedding": {"type": "rank_features"},
        }
    },
}

In [ ]:
# INDEX_NAME is an ALIAS pointing at a versioned index (document-chunks-v1, -v2, ...).
# To change the mapping of an existing library, use `python index_admin.py reindex`
# instead of recreating here -- that keeps the data and allows rollback.
#
# The index is shared by every document in the library (not just Gatsby), so
# recreating it wipes uploaded documents too. Only set True on purpose.
RECREATE_INDEX = False

if es.indices.exists(index=INDEX_NAME):  # true for both an alias and a legacy concrete index
    if es.indices.exists_alias(name=INDEX_NAME):
        targets = list(es.indices.get_alias(name=INDEX_NAME).keys())
    else:
        targets = [INDEX_NAME]  # legacy layout from before aliases
    if RECREATE_INDEX:
        print(f"Deleting {targets} behind '{INDEX_NAME}' and recreating (RECREATE_INDEX=True).")
        es.indices.delete(index=",".join(targets))  # deleting the index also drops its alias
    else:
        print(f"'{INDEX_NAME}' already exists ({targets}) -- leaving it as is (RECREATE_INDEX=False).")

if not es.indices.exists(index=INDEX_NAME):
    existing = es.cat.indices(index=f"{INDEX_NAME}-v*", format="json", h="index")
    versions = [int(r["index"].rsplit("-v", 1)[1]) for r in existing if r["index"].rsplit("-v", 1)[1].isdigit()]
    new_index = f"{INDEX_NAME}-v{max(versions, default=0) + 1}"
    es.indices.create(
        index=new_index,
        settings=INDEX_MAPPING["settings"],
        mappings=INDEX_MAPPING["mappings"],
        aliases={INDEX_NAME: {"is_write_index": True}},
    )
    print(f"Created index '{new_index}' with alias '{INDEX_NAME}'")

## Bulk index

Each chunk's `chunk_id` is used as the document `_id` (not an auto-generated id) -- this makes re-running this cell idempotent: re-indexing overwrites the same 247 documents instead of creating duplicates.

In [8]:
chunks = json.loads(EMBEDDED_JSON_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(chunks)} chunks to index")

# rank_features treats a literal "." in a token name as a nested-object path
# separator (e.g. "3.00" -> object "3" -> field "00"), which breaks indexing
# for any chunk containing such a token (bge-m3's sparse output includes
# punctuation/number subwords like ".", "3.00", "5.00"). Swap "." for a safe
# lookalike character so the token stays meaningful and searchable.
for c in chunks:
    c["sparse_embedding"] = {k.replace(".", "·"): v for k, v in c["sparse_embedding"].items()}


def make_actions(chunks):
    for chunk in chunks:
        yield {"_index": INDEX_NAME, "_id": chunk["chunk_id"], "_source": chunk}


success_count, errors = bulk(es, make_actions(chunks))
print(f"Indexed {success_count} documents, errors: {errors}")

Loaded 247 chunks to index
Indexed 247 documents, errors: []


### Sanity check

Confirm the document count matches, then run a kNN search using one chunk's own embedding as the query vector -- it should come back as the top hit with a cosine score of 1.0 (or extremely close to it), proving the vectors were stored and indexed correctly. A plain BM25 match on `text` checks the lexical side too.

In [9]:
es.indices.refresh(index=INDEX_NAME)
count = es.count(index=INDEX_NAME)["count"]
print(f"Document count in '{INDEX_NAME}': {count} (expected {len(chunks)})")

sample = chunks[10]
knn_query = {"field": "embedding", "query_vector": sample["embedding"], "k": 3, "num_candidates": 50}
resp = es.search(index=INDEX_NAME, knn=knn_query, source=["chunk_id", "chapter_title", "text"])
print(f"\nkNN self-similarity check for {sample['chunk_id']}:")
for hit in resp["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f}  {hit['_source']['chunk_id']}  (ch {hit['_source']['chapter_title']})")

resp2 = es.search(index=INDEX_NAME, query={"match": {"text": "green light"}}, source=["chunk_id", "chapter_title"], size=3)
print("\nBM25 match check for 'green light':")
for hit in resp2["hits"]["hits"]:
    print(f"  score={hit['_score']:.4f}  {hit['_source']['chunk_id']}  (ch {hit['_source']['chapter_title']})")

Document count in 'gatsby-chunks': 247 (expected 247)

kNN self-similarity check for great-gatsby-ch01-s01-c005:
  score=1.0003  great-gatsby-ch01-s01-c005  (ch I)
  score=0.9225  great-gatsby-ch01-s01-c004  (ch I)
  score=0.9214  great-gatsby-ch01-s01-c006  (ch I)

BM25 match check for 'green light':
  score=6.7016  great-gatsby-ch09-s05-c002  (ch IX)
  score=5.8933  great-gatsby-ch05-s01-c000  (ch V)
  score=5.7296  great-gatsby-ch09-s05-c001  (ch IX)
